# Extract Data locally?

To save costs, it often makes sense to use a library that converts a PDF to markdown and only send the markdown to the LLM instead of a large PDF. 


## Setup

In [4]:
import base64
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv(override=True)
client = OpenAI()

 # Invoice

In [5]:
INVOICE_PATH = Path("files/Anthropic_Invoice.pdf")

# Model

In [6]:
MODEL = "gpt-5.4-mini"

# Define Response Format

In [7]:
class DocumentAnalysis(BaseModel):
    is_invoice: bool
    vendor: str | None
    payment_date: str | None
    amount: float | None
    currency: str | None

# Define Instructions

In [8]:
INSTRUCTIONS = (
    "Decide whether the document is really an invoice. "
    "If it is not an invoice, set is_invoice to false and set "
    "vendor, payment_date, amount, and currency to null. "
    "If it is an invoice, set is_invoice to true and extract "
    "the invoice vendor, payment date, total amount, and currency. "
    "Use YYYY-MM-DD for the payment date. "
    "Use null for any missing invoice field."
)

# PDF to Markdown Conversion
Some popular PDF => Markdown conversion libraries: 
- [MinerU](https://github.com/opendatalab/MinerU)
- [Docling](https://docling-project.github.io/docling/)
- [Marker](https://github.com/datalab-to/marker)
- [PyMuPDF4LLM](https://github.com/pymupdf/pymupdf4llm)

Here we will use PyMuPDF4LLM because it is a lightweight solution. Docling, Marker, and MinerU are more powerful document-understanding systems, but they bring substantially more dependencies and possible model/runtime overhead.

In [9]:
import pymupdf4llm
from IPython.display import Markdown

invoice_markdown = pymupdf4llm.to_markdown(INVOICE_PATH)

display(Markdown(invoice_markdown))

## **Invoice** 

**==> picture [41 x 41] intentionally omitted <==**

**Invoice number** INV-000000 Date of issue May 9, 2026 Date due May 9, 2026 VAT Registration EU VAT: IE0000000XX 

## **Anthropic Ireland, Limited** 

## **Bill to** 

6th Floor South Bank House, Barrow Street, Lukas Lechner Dublin 4, DUBLIN, Ireland Example Street 1 DUBLIN 1000 Vienna Co. Dublin Austria Ireland student@example.com support@anthropic.com AT VAT ATU00000000 

## **€18.00 due May 9, 2026** 

## **Pay online** 

Reverse charge applies - Customer to account for tax under Article 196 of Directive 2006/112/EC. 

|Description|Qty<br>Unit price<br>Tax<br>Amount|
|---|---|
|Claude Pro<br>May 9�Jun 9, 2026|1<br>€18.00<br>0%<br>€18.00<br>�1�|
||Subtotal<br>€18.00|
||Total<br>€18.00|
||**Amount due**<br>**€18.00**|



Customer may be obliged to account for VAT on reverse charge basis. 

�1� Tax to be paid on reverse charge basis 

Page 1 of 1 



# LLM Call

In [ ]:
markdown_input = f"Extract the invoice data from this markdown:\n\n{invoice_markdown}"
response = client.responses.parse(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=markdown_input,
    text_format=DocumentAnalysis,
)

document_analysis = response.output_parsed

if document_analysis is not None:
    print(document_analysis.model_dump_json(indent=2))

{
  "is_invoice": true,
  "vendor": "Anthropic Ireland, Limited",
  "payment_date": "2026-05-09",
  "amount": 18.0,
  "currency": "EUR"
}


# Comparing Costs

We have used the `tiktoken` library in the first module to count the tokens of a prompt and calculate the costs of the LLM request. 

However, the `tiktoken` library is not sufficient here, as we can only count the tokens when  text is sent to the model. 

We can't use tiktoken when files are passed.

Instead, we have to use `client.responses.input_tokens` to determine how many input tokens a request will use before you send it to the model. 

See 
- [Counting Tokens](https://developers.openai.com/api/docs/guides/token-counting)
- [API Reference](https://developers.openai.com/api/reference/resources/responses/subresources/input_tokens/methods/count)

In [ ]:
INPUT_PRICE_PER_1M_TOKENS = 0.75
INPUT_PRICE_PER_TOKEN = 0.75 / 1_000_000

invoice_base64 = base64.b64encode(INVOICE_PATH.read_bytes()).decode("utf-8")

pdf_input = [
    {
        "role": "user",
        "content": [
            {
                "type": "input_file",
                "filename": INVOICE_PATH.name,
                "file_data": f"data:application/pdf;base64,{invoice_base64}",
            },
        ],
    }
]

pdf_token_count = client.responses.input_tokens.count(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=pdf_input, # type: ignore
)

markdown_token_count = client.responses.input_tokens.count(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=markdown_input,
)

pdf_input_cost = pdf_token_count.input_tokens * INPUT_PRICE_PER_TOKEN
markdown_input_cost = (
    markdown_token_count.input_tokens * INPUT_PRICE_PER_TOKEN
)
estimated_savings = pdf_input_cost - markdown_input_cost

print(f"PDF input tokens: {pdf_token_count.input_tokens:,}")
print(f"Markdown input tokens: {markdown_token_count.input_tokens:,}")
print(f"PDF input cost: ${pdf_input_cost:.6f}")
print(f"Markdown input cost: ${markdown_input_cost:.6f}")
print(f"Estimated input cost savings: ${estimated_savings:.6f}")

PDF input tokens: 1,282
Markdown input tokens: 399
PDF input cost: $0.000962
Markdown input cost: $0.000299
Estimated input cost savings: $0.000662
